# Clase 4: Evaluacion de Modelos de Machine Learning
## Comparacion y seleccion de modelos para problemas empresariales

---

### Objetivo de esta clase

En clases anteriores aprendimos a entrenar modelos de Machine Learning. Pero **entrenar un modelo no es suficiente**. Necesitamos saber:

- **¿Que tan bueno es el modelo?**
- **¿Es mejor que otro modelo?**
- **¿Que tipo de errores comete?**
- **¿Cual modelo deberia elegir para mi problema de negocio?**

### ¿Por que accuracy no siempre es suficiente?

Imaginemos que tenemos un modelo que predice si un paciente tiene una enfermedad rara que afecta al 1% de la poblacion. Si el modelo **siempre dice "no tiene la enfermedad"**, tendra un accuracy del 99%... pero **nunca detectara a los enfermos**.

Lo mismo ocurre en los negocios:

| Escenario | Consecuencia de un error |
|-----------|-------------------------|
| Detectar fraude bancario | Si no detectamos un fraude, el banco pierde dinero |
| Predecir churn de clientes | Si no detectamos que un cliente se va, lo perdemos |
| Filtrar spam en correos | Si marcamos un correo importante como spam, perdemos informacion |

Por eso necesitamos **multiples metricas** para evaluar un modelo.

---

### Metricas que aprenderemos hoy

| Metrica | ¿Que mide? |
|---------|------------|
| **Accuracy** | Porcentaje de predicciones correctas en general |
| **Precision** | De los que el modelo dijo "positivo", ¿cuantos realmente lo eran? |
| **Recall** | De los que realmente eran positivos, ¿cuantos detecto el modelo? |
| **F1 Score** | Promedio armonico entre Precision y Recall |
| **Confusion Matrix** | Tabla que muestra aciertos y errores del modelo |

### Modelos que compararemos

1. **Logistic Regression** - Modelo lineal, simple y rapido
2. **Random Forest** - Modelo basado en multiples arboles de decision
3. **Naive Bayes** - Modelo probabilistico, eficiente con pocos datos

---
## 1. Importar librerias

Comenzamos importando las librerias necesarias para el laboratorio.

In [ ]:
# Librerias para manejo de datos
import pandas as pd
import numpy as np

# Librerias para visualizacion
import matplotlib.pyplot as plt
import seaborn as sns

# Librerias de scikit-learn para modelos
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# Librerias de scikit-learn para evaluacion
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuracion de visualizacion
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Semilla para reproducibilidad
np.random.seed(42)

print("Librerias importadas correctamente")

---
## 2. Cargar dataset real

### Caso de negocio: Prediccion de Churn en telecomunicaciones

Una empresa de telecomunicaciones quiere **predecir si un cliente cancelara su servicio** (Customer Churn). Esto es importante porque:

- Adquirir un cliente nuevo cuesta **5 a 7 veces mas** que retener uno existente
- Si podemos predecir que un cliente se va a ir, podemos ofrecerle una promocion para retenerlo
- Un modelo que detecte churn a tiempo puede ahorrar millones a la empresa

Usaremos el dataset **Telco Customer Churn de IBM**, un dataset real con 7,043 clientes y 21 variables.

**Fuente:** [IBM Sample Data Sets - Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d)

In [ ]:
# Cargar el dataset desde el archivo CSV
df = pd.read_csv('Telco-Customer-Churn.csv')

print(f"Dataset cargado con {len(df)} registros y {len(df.columns)} columnas")
print(f"\nColumnas disponibles:")
print(df.columns.tolist())

In [ ]:
# Ver las primeras filas del dataset
df.head(10)

In [ ]:
# Informacion general del dataset
df.info()

In [ ]:
# Estadisticas descriptivas
df.describe(include='all')

### Limpieza de datos

El dataset tiene algunas particularidades que debemos resolver antes de continuar:

- La columna `TotalCharges` tiene valores vacios (espacios en blanco) que debemos convertir a numerico
- La columna `Churn` tiene valores "Yes"/"No" que debemos convertir a 1/0
- La columna `customerID` no es util para el modelo, la eliminaremos

In [ ]:
# Eliminar columna de ID (no aporta al modelo)
df = df.drop('customerID', axis=1)

# Convertir TotalCharges a numerico (tiene espacios vacios)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Eliminar filas con valores nulos (son solo 11 de 7043)
print(f"Valores nulos antes de limpiar: {df.isnull().sum().sum()}")
df = df.dropna()
print(f"Registros despues de limpiar: {len(df)}")

# Convertir Churn a numerico: Yes=1, No=0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"\nDistribucion de Churn:")
print(df['Churn'].value_counts())
print(f"\nPorcentaje de churn: {df['Churn'].mean()*100:.1f}%")

---
## 3. Exploracion de datos (EDA)

Antes de entrenar modelos, debemos **entender nuestros datos**. Vamos a visualizar las distribuciones y relaciones entre variables.

In [ ]:
# Distribucion de la variable objetivo: Churn
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Grafico de barras
churn_counts = df['Churn'].value_counts()
axes[0].bar(['No Churn (0)', 'Churn (1)'], churn_counts.values,
            color=['steelblue', 'salmon'])
axes[0].set_title('Distribucion de Churn')
axes[0].set_ylabel('Cantidad de clientes')

# Agregar etiquetas de valor
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Grafico de torta
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'],
            autopct='%1.1f%%', colors=['steelblue', 'salmon'],
            startangle=90)
axes[1].set_title('Proporcion de Churn')

plt.tight_layout()
plt.show()

In [ ]:
# Histogramas de las variables numericas principales
variables_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, var in enumerate(variables_numericas):
    # Histograma separado por churn
    axes[i].hist(df[df['Churn'] == 0][var], bins=20, alpha=0.6,
                 label='No Churn', color='steelblue')
    axes[i].hist(df[df['Churn'] == 1][var], bins=20, alpha=0.6,
                 label='Churn', color='salmon')
    axes[i].set_title(f'Distribucion de {var}')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Frecuencia')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Churn por tipo de contrato
fig, ax = plt.subplots(figsize=(8, 5))

contract_churn = df.groupby(['Contract', 'Churn']).size().unstack(fill_value=0)
contract_churn.plot(kind='bar', ax=ax, color=['steelblue', 'salmon'])
ax.set_title('Churn por tipo de contrato')
ax.set_xlabel('Tipo de Contrato')
ax.set_ylabel('Cantidad de clientes')
ax.legend(['No Churn', 'Churn'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlacion (solo variables numericas)
fig, ax = plt.subplots(figsize=(8, 6))

cols_numericas = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']
correlacion = df[cols_numericas].corr()
sns.heatmap(correlacion, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', ax=ax)
ax.set_title('Matriz de Correlacion')

plt.tight_layout()
plt.show()

---
## 4. Preparacion de datos

Antes de entrenar los modelos, necesitamos:

1. **Codificar variables categoricas**: Los modelos de ML trabajan con numeros, no con texto. Debemos convertir columnas como `gender`, `Contract`, `InternetService`, etc. a valores numericos.
2. **Separar variables predictoras (X) y variable objetivo (y)**
3. **Dividir en conjunto de entrenamiento y prueba**: Usamos parte de los datos para entrenar y otra parte para evaluar.
4. **Escalar las variables numericas**: Las variables tienen escalas muy diferentes (por ejemplo, `tenure` va de 0 a 72 y `TotalCharges` de 0 a 8000+). Si no las escalamos, los modelos dan mas importancia a las variables con valores grandes.

In [ ]:
# Identificar columnas categoricas
cols_categoricas = df.select_dtypes(include='object').columns.tolist()
print(f"Columnas categoricas ({len(cols_categoricas)}):")
print(cols_categoricas)

# Codificar usando one-hot encoding
df_encoded = pd.get_dummies(df, columns=cols_categoricas, drop_first=True)

print(f"\nColumnas despues del encoding ({len(df_encoded.columns)}):")
print(df_encoded.columns.tolist())
df_encoded.head()

In [ ]:
# Separar variables predictoras (X) y variable objetivo (y)
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f"Variables predictoras (X): {X.shape[1]} columnas")
print(f"Variable objetivo (y): Churn")
print(f"\nForma de X: {X.shape}")
print(f"Forma de y: {y.shape}")

In [ ]:
# Dividir en conjunto de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos de entrenamiento: {X_train.shape[0]} registros")
print(f"Datos de prueba: {X_test.shape[0]} registros")
print(f"\nDistribucion de Churn en entrenamiento:")
print(y_train.value_counts())
print(f"\nDistribucion de Churn en prueba:")
print(y_test.value_counts())

In [ ]:
# Escalar las variables para que todas tengan la misma magnitud
# StandardScaler transforma cada variable para que tenga media=0 y desviacion estandar=1
scaler = StandardScaler()

# Importante: ajustamos el scaler SOLO con datos de entrenamiento para evitar data leakage
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Datos escalados correctamente")
print(f"\nEjemplo - Media de las primeras 3 columnas en entrenamiento:")
print(X_train.iloc[:, :3].mean().round(4))
print(f"\nEjemplo - Desviacion estandar de las primeras 3 columnas:")
print(X_train.iloc[:, :3].std().round(4))

---
## 5. Entrenamiento de modelos

Vamos a entrenar **tres modelos diferentes** para luego compararlos:

| Modelo | Descripcion |
|--------|-------------|
| **Logistic Regression** | Modelo lineal que estima probabilidades. Simple, rapido e interpretable. |
| **Random Forest** | Conjunto de arboles de decision que votan por la prediccion final. Mas complejo pero generalmente mas preciso. |
| **Naive Bayes** | Modelo probabilistico basado en el teorema de Bayes. Rapido y funciona bien con pocos datos. |

**Nota:** Como nuestro dataset tiene mas clientes que NO hacen churn (~73%) que los que SI lo hacen (~27%), usamos `class_weight='balanced'` en Logistic Regression y Random Forest. Esto le indica al modelo que le de mas importancia a la clase minoritaria (churn) durante el entrenamiento.

In [ ]:
# --- Modelo 1: Logistic Regression ---
modelo_lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)

print("Modelo 1: Logistic Regression - Entrenado correctamente")

In [ ]:
# --- Modelo 2: Random Forest ---
modelo_rf = RandomForestClassifier(n_estimators=50, random_state=42, class_weight='balanced')
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)

print("Modelo 2: Random Forest (n_estimators=50) - Entrenado correctamente")

In [ ]:
# --- Modelo 3: Naive Bayes ---
modelo_nb = GaussianNB()
modelo_nb.fit(X_train, y_train)
y_pred_nb = modelo_nb.predict(X_test)

print("Modelo 3: Naive Bayes - Entrenado correctamente")

---
## 6. Evaluacion de modelos

Ahora vamos a evaluar cada modelo usando las metricas que aprendimos.

### Recordemos que significa cada metrica:

- **Accuracy**: ¿Que porcentaje de todas las predicciones fue correcto?
- **Precision**: Cuando el modelo dice "este cliente hara churn", ¿con que frecuencia tiene razon?
- **Recall**: De todos los clientes que realmente hicieron churn, ¿cuantos detecto el modelo?
- **F1 Score**: Balance entre Precision y Recall (util cuando las clases estan desbalanceadas)

### ¿Que es la Confusion Matrix?

```
                    Prediccion
                  No Churn  |  Churn
Real  No Churn |    TN     |   FP    |
      Churn    |    FN     |   TP    |
```

- **TN (True Negative)**: El modelo predijo que el cliente **no se iria**, y efectivamente **se quedo**. Acierto.
- **TP (True Positive)**: El modelo predijo que el cliente **se iria**, y efectivamente **se fue**. Acierto.
- **FP (False Positive)**: El modelo predijo que el cliente **se iria**, pero en realidad **se quedo**. Es una falsa alarma: gastamos recursos de retencion en alguien que no lo necesitaba.
- **FN (False Negative)**: El modelo predijo que el cliente **se quedaria**, pero en realidad **se fue**. Es el error mas costoso: perdimos al cliente sin intentar retenerlo.

In [ ]:
# Funcion para evaluar un modelo y mostrar sus metricas
def evaluar_modelo(nombre, y_real, y_predicho):
    """Calcula y muestra las metricas de evaluacion de un modelo."""
    print(f"{'='*50}")
    print(f"  {nombre}")
    print(f"{'='*50}")
    print(f"  Accuracy:  {accuracy_score(y_real, y_predicho):.4f}")
    print(f"  Precision: {precision_score(y_real, y_predicho, zero_division=0):.4f}")
    print(f"  Recall:    {recall_score(y_real, y_predicho, zero_division=0):.4f}")
    print(f"  F1 Score:  {f1_score(y_real, y_predicho, zero_division=0):.4f}")
    print(f"\n  Reporte de clasificacion:")
    print(classification_report(y_real, y_predicho, target_names=['No Churn', 'Churn'], zero_division=0))
    return {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_real, y_predicho),
        'Precision': precision_score(y_real, y_predicho, zero_division=0),
        'Recall': recall_score(y_real, y_predicho, zero_division=0),
        'F1 Score': f1_score(y_real, y_predicho, zero_division=0)
    }

In [ ]:
# Evaluar los tres modelos
resultados = []

resultados.append(evaluar_modelo('Logistic Regression', y_test, y_pred_lr))
resultados.append(evaluar_modelo('Random Forest', y_test, y_pred_rf))
resultados.append(evaluar_modelo('Naive Bayes', y_test, y_pred_nb))

In [ ]:
# Tabla comparativa de resultados
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.set_index('Modelo')

print("\nTABLA COMPARATIVA DE MODELOS")
print("=" * 60)
df_resultados.round(4)

In [ ]:
# Conclusion automatica basada en los resultados
mejor_accuracy = df_resultados['Accuracy'].idxmax()
mejor_precision = df_resultados['Precision'].idxmax()
mejor_recall = df_resultados['Recall'].idxmax()
mejor_f1 = df_resultados['F1 Score'].idxmax()

print("CONCLUSION: ¿Cual modelo es mejor?")
print("=" * 60)
print(f"\n  Mejor Accuracy:  {mejor_accuracy} ({df_resultados.loc[mejor_accuracy, 'Accuracy']:.4f})")
print(f"  Mejor Precision: {mejor_precision} ({df_resultados.loc[mejor_precision, 'Precision']:.4f})")
print(f"  Mejor Recall:    {mejor_recall} ({df_resultados.loc[mejor_recall, 'Recall']:.4f})")
print(f"  Mejor F1 Score:  {mejor_f1} ({df_resultados.loc[mejor_f1, 'F1 Score']:.4f})")

print(f"\n{'='*60}")
print(f"  RECOMENDACION PARA EL NEGOCIO")
print(f"{'='*60}")
print(f"\n  Para el problema de churn, donde perder un cliente sin")
print(f"  detectarlo es muy costoso, la metrica mas importante es Recall.")
print(f"\n  El modelo con mejor Recall es: ** {mejor_recall} **")
print(f"  Detecta el {df_resultados.loc[mejor_recall, 'Recall']*100:.1f}% de los clientes que realmente se van.")
print(f"\n  Si buscamos un balance entre Precision y Recall,")
print(f"  el mejor modelo por F1 Score es: ** {mejor_f1} **")

---
## 7. Visualizacion de resultados

Vamos a visualizar las matrices de confusion y comparar las metricas de los tres modelos.

In [ ]:
# Matrices de confusion para los tres modelos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

modelos = [
    ('Logistic Regression', y_pred_lr),
    ('Random Forest', y_pred_rf),
    ('Naive Bayes', y_pred_nb)
]

for i, (nombre, y_pred) in enumerate(modelos):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    axes[i].set_title(f'Confusion Matrix\n{nombre}')
    axes[i].set_ylabel('Valor Real')
    axes[i].set_xlabel('Prediccion')

plt.tight_layout()
plt.show()

In [ ]:
# Comparacion de metricas entre modelos (grafico de barras agrupadas)
fig, ax = plt.subplots(figsize=(12, 6))

metricas = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(metricas))
ancho = 0.25

# Barras para cada modelo
bars1 = ax.bar(x - ancho, df_resultados.loc['Logistic Regression'].values,
               ancho, label='Logistic Regression', color='steelblue')
bars2 = ax.bar(x, df_resultados.loc['Random Forest'].values,
               ancho, label='Random Forest', color='forestgreen')
bars3 = ax.bar(x + ancho, df_resultados.loc['Naive Bayes'].values,
               ancho, label='Naive Bayes', color='salmon')

# Agregar valores sobre las barras
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metrica')
ax.set_ylabel('Valor')
ax.set_title('Comparacion de Metricas entre Modelos')
ax.set_xticks(x)
ax.set_xticklabels(metricas)
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

---
## 8. Interpretacion empresarial

### ¿Que significan estos resultados para la empresa?

Ahora que tenemos las metricas, debemos interpretarlas desde la perspectiva del negocio.

### Preguntas clave:

#### 1. ¿Que pasa si el modelo NO detecta un cliente que va a hacer churn? (False Negative)

- El cliente se va **sin que la empresa haga nada para retenerlo**
- La empresa pierde ingresos recurrentes
- El costo de adquirir un nuevo cliente es mucho mayor
- **Un False Negative es MUY costoso** en este problema

#### 2. ¿Que pasa si el modelo dice que un cliente hara churn, pero no es cierto? (False Positive)

- La empresa ofrece una promocion o descuento a un cliente que no se iba a ir
- Se gasta dinero en retencion innecesaria, pero **no se pierde al cliente**
- Es un costo, pero menor que perder al cliente

#### 3. ¿Que metrica es mas importante para esta empresa?

En este caso, **Recall es la metrica mas importante** porque:
- Queremos detectar la mayor cantidad posible de clientes que se van a ir
- Es mejor tener algunas falsas alarmas que dejar ir clientes sin intentar retenerlos
- Un modelo con alto Recall captura mas casos reales de churn

#### 4. ¿En que casos es mejor priorizar Precision?

- Cuando el costo de la accion es alto (ejemplo: tratamientos medicos costosos)
- Cuando no queremos molestar a clientes que estan satisfechos
- Cuando los recursos para retencion son muy limitados

### Regla general:

| Priorizar Recall | Priorizar Precision |
|-------------------|---------------------|
| El costo de NO detectar es alto | El costo de actuar sin razon es alto |
| Deteccion de fraude | Filtro de spam |
| Diagnostico de enfermedades | Recomendacion de productos |
| **Prediccion de churn** | Aprobacion de creditos |

---
## 9. EJERCICIO PARA LOS ESTUDIANTES

### Responde las siguientes preguntas analizando los resultados obtenidos:

**Pregunta 1:** ¿Cual modelo tuvo el mejor Recall? ¿Por que es importante esto para la empresa de telecomunicaciones?

**Pregunta 2:** ¿Cual modelo usarias si fueras el gerente de la empresa? Justifica tu respuesta con las metricas.

**Pregunta 3:** ¿Que pasaria si el churn fuera muy raro (por ejemplo, solo el 2% de los clientes)? ¿Accuracy seguiria siendo una buena metrica? ¿Por que?

**Pregunta 4:** Observa las matrices de confusion. ¿Cual modelo tuvo mas False Negatives? ¿Que implicacion tiene esto para el negocio?

**Pregunta 5:** Si la empresa solo tiene presupuesto para llamar a 50 clientes para ofrecerles una promocion, ¿que metrica priorizarias y por que?

In [ ]:
# Espacio para las respuestas de los estudiantes
# Escribe tus respuestas como comentarios aqui:

# Pregunta 1:
# 

# Pregunta 2:
# 

# Pregunta 3:
# 

# Pregunta 4:
# 

# Pregunta 5:
# 

---
## 10. Actividad adicional: Mejorar el modelo Random Forest

### Instrucciones:

El modelo Random Forest que entrenamos usa `n_estimators=50` (50 arboles de decision).

**Tu tarea:**
1. Cambia `n_estimators` de 50 a 200
2. Entrena el nuevo modelo
3. Evalua sus metricas
4. Compara los resultados con el modelo original

¿Mejoro el modelo? ¿Cambio significativamente alguna metrica?

In [ ]:
# --- Actividad: Random Forest con n_estimators=200 ---

# Paso 1: Crear el nuevo modelo con 200 arboles
modelo_rf_200 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')

# Paso 2: Entrenar el modelo
modelo_rf_200.fit(X_train, y_train)

# Paso 3: Hacer predicciones
y_pred_rf_200 = modelo_rf_200.predict(X_test)

print("Random Forest con n_estimators=200 - Entrenado correctamente")

In [ ]:
# Paso 4: Evaluar y comparar
print("COMPARACION: Random Forest 50 arboles vs 200 arboles")
print("=" * 60)

resultado_rf_200 = evaluar_modelo('Random Forest (200 arboles)', y_test, y_pred_rf_200)

print("\n" + "=" * 60)
print("  COMPARACION DIRECTA")
print("=" * 60)

comparacion = pd.DataFrame([
    resultados[1],  # Random Forest original (50 arboles)
    resultado_rf_200
])
comparacion = comparacion.set_index('Modelo')
comparacion.round(4)

In [ ]:
# Visualizar la comparacion
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Matrices de confusion comparativas
for i, (nombre, y_pred) in enumerate([
    ('Random Forest (50 arboles)', y_pred_rf),
    ('Random Forest (200 arboles)', y_pred_rf_200)
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[i],
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    axes[i].set_title(f'Confusion Matrix\n{nombre}')
    axes[i].set_ylabel('Valor Real')
    axes[i].set_xlabel('Prediccion')

plt.tight_layout()
plt.show()

---
## 11. Conclusiones de la clase

### Lo que aprendimos hoy:

1. **Accuracy no es suficiente** para evaluar un modelo, especialmente cuando las clases estan desbalanceadas.

2. **Precision y Recall** nos dan informacion complementaria:
   - Precision = ¿que tan confiable es cuando dice "positivo"?
   - Recall = ¿cuantos positivos logra detectar?

3. **F1 Score** es util cuando necesitamos un balance entre Precision y Recall.

4. **La matriz de confusion** nos permite ver exactamente donde se equivoca el modelo.

5. **La eleccion del modelo depende del problema de negocio**:
   - Para churn: priorizamos Recall (no queremos perder clientes sin hacer nada)
   - Para spam: priorizamos Precision (no queremos filtrar correos importantes)

6. **Comparar modelos** nos permite elegir el mas adecuado para nuestro caso especifico.

### Para la proxima clase:

Aprenderemos sobre **validacion cruzada** y como evitar el **sobreajuste (overfitting)** de los modelos.